In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

## 1. Project paths

In [1]:
BASE_DIR = Path(__file__).resolve().parents[1]

DATA_PATH = BASE_DIR / "data" / "diabetic_data.csv"
MAPPING_PATH = BASE_DIR / "data" / "IDS_mapping.csv"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


NameError: name 'Path' is not defined

### 2. Loading the main dataset

In [ ]:
def load_data():
    """Load the original diabetes hospital encounters dataset."""

    df = pd.read_csv(
        DATA_PATH,
        dtype={
            "diag_1": str,
            "diag_2": str,
            "diag_3": str
        }
    )

    print("Original dataset shape:", df.shape)

    return df


### 3. Load the ID mapping file

In [ ]:
def load_id_mappings():
    """
    Read the mapping sections in IDS_mapping.csv.

    The file contains mappings for:
    - admission_type_id
    - discharge_disposition_id
    - admission_source_id
    """

    if not MAPPING_PATH.exists():
        raise FileNotFoundError(
            f"Mapping file not found: {MAPPING_PATH}"
        )

    # Read the mapping file as raw rows because it contains
    # multiple mapping sections separated by blank rows.
    raw = pd.read_csv(
        MAPPING_PATH,
        header=None,
        dtype=str,
        keep_default_na=False
    )

    mappings = {}
    current_section = None

    valid_sections = {
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id"
    }

    for _, row in raw.iterrows():

        first = str(row.iloc[0]).strip()

        second = (
            str(row.iloc[1]).strip()
            if len(row) > 1
            else ""
        )

        # Skip blank rows
        if first == "":
            current_section = None
            continue

        # Detect the start of a new mapping section
        if first in valid_sections:
            current_section = first
            mappings[current_section] = {}
            continue

        # Store each ID and its corresponding description
        if current_section is not None and second != "":
            mappings[current_section][first] = second

    # Check that all expected mapping sections were loaded
    missing_sections = valid_sections - set(mappings.keys())

    if missing_sections:
        raise ValueError(
            "The following mapping sections were not found: "
            f"{missing_sections}"
        )

    print("ID mapping sections loaded successfully.")

    for section, mapping in mappings.items():
        print(f"{section}: {len(mapping)} mappings")

    return mappings

### 4. Map the ID columns to descriptions

In [ ]:
def map_id_columns(df, mappings):
    """
    Replace numeric hospital ID codes with descriptive
    categorical values using IDS_mapping.csv.
    """

    columns_to_map = {
        "admission_type_id": (
            "admission_type"
        ),
        "discharge_disposition_id": (
            "discharge_disposition"
        ),
        "admission_source_id": (
            "admission_source"
        )
    }

    for id_column, description_column in columns_to_map.items():

        if id_column not in df.columns:
            raise ValueError(
                f"Expected column '{id_column}' was not found."
            )

        mapping = mappings[id_column]

        # Convert IDs to strings so they match the mapping file
        id_values = (
            df[id_column]
            .astype("string")
            .str.strip()
        )

        # Replace the ID codes with their descriptions
        df[description_column] = id_values.map(mapping)

        # Keep the original missing values as missing
        df.loc[
            id_values.isna() | id_values.eq(""),
            description_column
        ] = pd.NA

        # Check whether any non-missing ID could not be mapped
        unmapped = (
            id_values.notna()
            & id_values.ne("")
            & df[description_column].isna()
        )

        if unmapped.any():
            unknown_ids = sorted(
                id_values.loc[unmapped].unique().tolist()
            )

            raise ValueError(
                f"Unmapped values in {id_column}: {unknown_ids}"
            )

        # Remove the original numeric ID column
        df = df.drop(columns=[id_column])

    print("Hospital ID columns mapped successfully.")

    return df

### 5.data cleaning and creating the target

In [ ]:
def clean_data(df, mappings):
    """Clean the dataset and create the binary target."""

    # Replace question marks used as missing-value placeholders
    df = df.replace("?", pd.NA)

    # Map the admission, discharge and source IDs
    df = map_id_columns(df, mappings)

    # Create binary readmission target
    # 1 = readmitted within 30 days
    # 0 = no recorded readmission within 30 days
    df["target"] = (
        df["readmitted"] == "<30"
    ).astype(int)

    # Remove the original readmission category
    df = df.drop(columns=["readmitted"])

    # Remove encounter identifier and constant columns
    df = df.drop(
        columns=[
            "encounter_id",
            "examide",
            "citoglipton"
        ],
        errors="ignore"
    )

    print("Cleaned dataset shape:", df.shape)

    print("\nTarget distribution:")
    print(df["target"].value_counts())

    return df

### 6.  patient-grouped training and testing sets

In [ ]:
def split_data(df):
    """
    Split the dataset by patient to reduce data leakage
    caused by repeated encounters for the same patient.
    """

    groups = df["patient_nbr"]

    # identifiers excluded from the predictors
    X = df.drop(
        columns=[
            "patient_nbr",
            "target"
        ],
        errors="ignore"
    )

    y = df["target"]

    #  80/20 grouped split
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    # Save the split datasets
    X_train.to_csv(
        OUTPUT_DIR / "X_train.csv",
        index=False
    )

    X_test.to_csv(
        OUTPUT_DIR / "X_test.csv",
        index=False
    )

    y_train.to_frame(name="target").to_csv(
        OUTPUT_DIR / "y_train.csv",
        index=False
    )

    y_test.to_frame(name="target").to_csv(
        OUTPUT_DIR / "y_test.csv",
        index=False
    )

    # Checking for patient overlap
    train_patients = set(groups.iloc[train_idx])
    test_patients = set(groups.iloc[test_idx])

    overlap = train_patients.intersection(test_patients)

    print("\nTraining records:", len(X_train))
    print("Testing records:", len(X_test))

    print(
        "Patients appearing in both sets:",
        len(overlap)
    )

    if len(overlap) > 0:
        raise ValueError(
            "Patient overlap detected between training "
            "and testing sets."
        )

    return X_train, X_test, y_train, y_test


### 7. Run the preprocessing workflow

In [ ]:
def main():

    df = load_data()

    mappings = load_id_mappings()

    df = clean_data(
        df,
        mappings
    )

    split_data(df)

    print("\nPreprocessing completed successfully.")


if __name__ == "__main__":
    main()